# sr-diffusion x4 Colab demo

Upload an image and run x4 super-resolution with the public `jwheo/sr-diffusion` checkpoints.

1. Use **Runtime -> Change runtime type -> GPU**.
2. Choose options in **Demo settings** below.
3. Use **Runtime -> Run all**, then upload one image when prompted.

The recommended model is the deterministic Residual Refiner v2. It is the strongest current public choice, runs comfortably on T4, and avoids the slower diffusion sampling path. Checkpoints are research prototypes under a non-commercial license.


In [ ]:
#@title Demo settings { display-mode: "form" }
MODEL_PRESET = 'Recommended quality - Residual Refiner v2' #@param ['Recommended quality - Residual Refiner v2', 'Sharper diffusion comparison - XL Edge', 'Smaller diffusion comparison - Stage 4 v2', 'Mild diffusion comparison - Stage 4']
UPLOAD_IMAGE = True #@param {type:"boolean"}
INPUT_TYPE = 'Low-resolution image to upscale' #@param ['Low-resolution image to upscale', 'High-resolution image for controlled test']
CORRECTION_STYLE = 'Full correction - best average quality' #@param ['Full correction - best average quality', 'Balanced correction - fewer regressions', 'Conservative correction - safest']
USE_TILING = True #@param {type:"boolean"}

MODEL_OPTIONS = {
    'Recommended quality - Residual Refiner v2': 'residual_refiner_v2',
    'Sharper diffusion comparison - XL Edge': 'photo100k_xl_edge_b16',
    'Smaller diffusion comparison - Stage 4 v2': 'photo100k_v2_stage4',
    'Mild diffusion comparison - Stage 4': 'photo100k_stage4',
}
MODEL_VARIANT = MODEL_OPTIONS[MODEL_PRESET]
USE_UPLOAD = bool(UPLOAD_IMAGE)
INPUT_MODE = 'lr' if INPUT_TYPE == 'Low-resolution image to upscale' else 'hr'
RESIDUAL_STRENGTH = {
    'Full correction - best average quality': 1.0,
    'Balanced correction - fewer regressions': 0.75,
    'Conservative correction - safest': 0.5,
}[CORRECTION_STYLE]

import torch

print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    GPU_TOTAL_GB = props.total_memory / 1024 ** 3
    print('gpu:', torch.cuda.get_device_name(0))
    print('gpu memory:', f'{GPU_TOTAL_GB:.1f} GB')
else:
    raise RuntimeError('GPU runtime is required. In Colab, use Runtime -> Change runtime type -> GPU.')


## 1. Clone the repo and install lightweight dependencies

Colab already provides PyTorch, so this avoids reinstalling `torch`.


In [ ]:
#@title Prepare repository { display-mode: "form" }
import os
import subprocess
from pathlib import Path

os.chdir('/content')
repo = Path('sr-diffusion')
if not repo.exists():
    subprocess.run(['git', 'clone', 'https://github.com/BitIntx/sr-diffusion.git'], check=True)
os.chdir(repo)
subprocess.run(['git', 'pull', '--ff-only'], check=True)
print('repo:', Path.cwd())


In [ ]:
#@title Install dependencies { display-mode: "form" }
%pip -q install pyyaml pillow huggingface_hub
%pip -q install -e . --no-deps


## 2. Download the selected model

The selected model and required checkpoints are downloaded automatically.


In [ ]:
#@title Download selected model { display-mode: "form" }
import subprocess
from pathlib import Path

REPO_ID = 'jwheo/sr-diffusion'
try:
    GPU_TOTAL_GB
except NameError:
    GPU_TOTAL_GB = 0.0

AVAILABLE_VARIANTS = ('residual_refiner_v2', 'photo100k_xl_edge_b16', 'photo100k_v2_stage4', 'photo100k_stage4', 'prototype_stage4', 'photo100k_v2_stage3')

COMMON_FILES = [
    'LICENSE',
    'CHECKPOINT_LICENSE.md',
    'checkpoints/stage1_autoencoder_best_eval_recon.pt',
]

VARIANTS = {
    'residual_refiner_v2': {
        'runner': 'residual_refiner',
        'config': 'configs/hf/residual_refiner_stage2_xl_photo_detail_v2.yaml',
        'files': [
            'checkpoints/stage2_photo100k_v3_noise_xl_b64_step_0072000.pt',
            'checkpoints/residual_refiner_stage2_xl_photo_detail_v2_best39000.pt',
        ],
        'note': 'Default deterministic decoded-detail refiner v2. Selected step 39000 improves condition-only mean PSNR on all tested presets.',
    },
    'photo100k_xl_edge_b16': {
        'runner': 'diffusion',
        'config': 'configs/hf/diffusion_photo100k_xl_stage4_condition_v3_edge_b16.yaml',
        'files': [
            'checkpoints/stage2_photo100k_v3_noise_xl_b64_step_0072000.pt',
            'checkpoints/stage4_photo100k_xl_edge_b16_best_eval_condition_decoded.pt',
        ],
        'note': 'Latest public XL Stage 4 edge-loss checkpoint. v3 val100: SR 23.0793 PSNR vs bicubic 22.3599 (+0.7195 dB). T4 can try it with conservative settings, but v2 is the fallback if it is too slow or OOMs.',
    },
    'photo100k_v2_stage4': {
        'runner': 'diffusion',
        'config': 'configs/hf/diffusion_photo100k_stage4_condition_v2.yaml',
        'files': [
            'checkpoints/stage2_photo100k_v2_b64_best_eval_latent.pt',
            'checkpoints/stage4_photo100k_condition_v2_b32_best_eval_condition_decoded.pt',
        ],
        'note': 'Smaller public denoise/sharpening checkpoint: photo100k v2 Stage 4 condition-start.',
    },
    'photo100k_stage4': {
        'runner': 'diffusion',
        'config': 'configs/hf/diffusion_photo100k_stage4_condition.yaml',
        'files': [
            'checkpoints/stage2_photo100k_b64_best_eval_latent.pt',
            'checkpoints/stage4_photo100k_condition_b32_best_eval_condition_decoded.pt',
        ],
        'note': 'Milder photo100k Stage 4 condition-start checkpoint.',
    },
    'prototype_stage4': {
        'runner': 'diffusion',
        'config': 'configs/hf/diffusion_stage4_condition.yaml',
        'files': [
            'checkpoints/stage2_latent_pretrain_best_eval_latent.pt',
            'checkpoints/stage4_condition_b32_best_eval_condition_decoded.pt',
        ],
        'note': 'Older 10k Stage 4 condition-start prototype.',
    },
    'photo100k_v2_stage3': {
        'runner': 'diffusion',
        'config': 'configs/hf/diffusion_photo100k_v2.yaml',
        'files': [
            'checkpoints/stage2_photo100k_v2_b64_best_eval_latent.pt',
            'checkpoints/stage3_photo100k_v2_b32_best_eval_noise.pt',
        ],
        'note': 'Previous v2 Stage 3 checkpoint, kept for comparison.',
    },
}

if MODEL_VARIANT not in VARIANTS:
    raise ValueError(f'MODEL_VARIANT must be one of {AVAILABLE_VARIANTS}, got {MODEL_VARIANT!r}')
selected = VARIANTS[MODEL_VARIANT]
files_to_download = [*COMMON_FILES, *selected['files']]
cmd = ['python', 'scripts/download_hf_checkpoints.py', '--repo-id', REPO_ID]
for filename in files_to_download:
    cmd += ['--file', filename]

print('variant:', MODEL_VARIANT)
print('gpu memory:', f'{GPU_TOTAL_GB:.1f} GB')
print(selected['note'])
if MODEL_VARIANT == 'photo100k_xl_edge_b16' and GPU_TOTAL_GB and GPU_TOTAL_GB < 20:
    print('T4/small-GPU mode: XL will run with fp16 fallback, tile batch size 1, and fewer default steps.')
print('config:', selected['config'])
subprocess.run(cmd, check=True)

missing = [filename for filename in files_to_download if not Path(filename).exists()]
if missing:
    raise FileNotFoundError(f'Missing downloaded files: {missing}')
CONFIG_PATH = selected['config']
if not Path(CONFIG_PATH).exists():
    raise FileNotFoundError(CONFIG_PATH)
print('downloaded files:', len(files_to_download))


## 3. Upload an input image

Run all prompts for an image by default. Low-resolution input is upscaled directly; controlled-test input is cropped and degraded before evaluation.


In [ ]:
#@title Upload image { display-mode: "form" }
from pathlib import Path
from PIL import Image, ImageDraw, ImageFilter
import numpy as np

if USE_UPLOAD:
    from google.colab import files
    uploaded = files.upload()
    INPUT_IMAGE = next(iter(uploaded))
else:
    demo_dir = Path('demo_inputs')
    demo_dir.mkdir(exist_ok=True)
    hr_path = demo_dir / 'synthetic_hr_768.png'
    h = w = 768
    yy, xx = np.mgrid[0:h, 0:w]
    base = np.zeros((h, w, 3), dtype=np.uint8)
    base[..., 0] = np.clip(40 + xx * 180 / w, 0, 255)
    base[..., 1] = np.clip(30 + yy * 190 / h, 0, 255)
    base[..., 2] = np.clip(180 - (xx + yy) * 70 / (w + h), 0, 255)
    image = Image.fromarray(base, mode='RGB')
    draw = ImageDraw.Draw(image)
    for i in range(24, 768, 56):
        draw.line((i, 24, 768 - i // 3, 744), fill=(245, 245, 245), width=2)
        draw.rectangle((i, i // 2, i + 38, i // 2 + 38), outline=(20, 20, 20), width=2)
    draw.text((24, 24), 'sr-diffusion smoke test', fill=(255, 255, 255))
    image = image.filter(ImageFilter.UnsharpMask(radius=1.0, percent=120, threshold=3))
    image.save(hr_path)
    if INPUT_MODE == 'hr':
        INPUT_IMAGE = str(hr_path)
    else:
        lr_path = demo_dir / 'synthetic_lr_192.png'
        image.resize((192, 192), Image.Resampling.BICUBIC).save(lr_path)
        INPUT_IMAGE = str(lr_path)

print('input mode:', INPUT_MODE)
print('input:', INPUT_IMAGE)


## 4. Run x4 SR

Residual refiner v2 is deterministic and does not use DDIM steps. For diffusion variants, keep `STEPS = 32` on A100/L4 or larger. On T4, XL starts at 16 DDIM steps for responsiveness and memory headroom.


In [ ]:
#@title Run x4 upscaling { display-mode: "form" }
import subprocess
from pathlib import Path

TILE_OVERLAP = 32
IS_REFINER = selected['runner'] == 'residual_refiner'
if IS_REFINER:
    TILE_BATCH_SIZE = 4 if GPU_TOTAL_GB < 20 else 8
else:
    TILE_BATCH_SIZE = 1 if MODEL_VARIANT == 'photo100k_xl_edge_b16' or GPU_TOTAL_GB < 20 else 4
OUTPUT_DIR = Path(f'outputs/colab_demo_{MODEL_VARIANT}')
STEPS = 16 if MODEL_VARIANT == 'photo100k_xl_edge_b16' and GPU_TOTAL_GB and GPU_TOTAL_GB < 20 else 32
SEED = 123
PROGRESS_EVERY = 4

input_flag = '--input-lr' if INPUT_MODE == 'lr' else '--input-hr'
runner_script = 'tools/infer/infer_residual_refiner.py' if IS_REFINER else 'tools/infer/infer_diffusion.py'
RESULT_FILE = 'refined.png' if IS_REFINER else 'sr_00.png'
cmd = [
    'python', '-u', runner_script,
    '--config', CONFIG_PATH,
    input_flag, INPUT_IMAGE,
    '--output-dir', str(OUTPUT_DIR),
    '--seed', str(SEED),
]
if not IS_REFINER:
    cmd += ['--steps', str(STEPS), '--progress-every', str(PROGRESS_EVERY)]
else:
    cmd += ['--residual-strength', str(RESIDUAL_STRENGTH)]
if INPUT_MODE == 'lr' and USE_TILING:
    cmd += ['--tile', '--tile-overlap', str(TILE_OVERLAP), '--tile-batch-size', str(TILE_BATCH_SIZE)]
print('steps:', 'deterministic' if IS_REFINER else STEPS)
print('tile batch size:', TILE_BATCH_SIZE)
if IS_REFINER:
    print('correction strength:', RESIDUAL_STRENGTH)
print(' '.join(cmd))
subprocess.run(cmd, check=True)


## 5. View and download the result

In [ ]:
#@title View result { display-mode: "form" }
from IPython.display import display
from PIL import Image
from pathlib import Path

out = Path(f'outputs/colab_demo_{MODEL_VARIANT}')
print('variant:', MODEL_VARIANT)
print('LR input')
display(Image.open(out / 'input_lr.png'))
gt = out / 'gt_hr.png'
if gt.exists():
    print('GT HR')
    display(Image.open(gt))
condition = out / 'condition.png'
if condition.exists():
    print('Stage 2 condition')
    display(Image.open(condition))
print('SR output')
display(Image.open(out / RESULT_FILE))


In [ ]:
#@title Download result { display-mode: "form" }
from google.colab import files
files.download(str(Path(f'outputs/colab_demo_{MODEL_VARIANT}') / RESULT_FILE))


## Optional: compare variants

Choose a different model in **Demo settings**, then rerun sections 2, 4, and 5.

Use `residual_refiner_v2` for the fastest conservative output and the best current cross-preset condition improvement. Use `photo100k_xl_edge_b16` for the more aggressive diffusion denoise/sharpening comparison, or `photo100k_stage4` for cleaner/milder inputs.
